# RM residuals by Gaussian-environment quartile

The first panels are data only: three spectroscopic-richness bins formed by merging the highest two of the original four global quartiles, three redshift bins, and four environment quartiles within each cell. The observed mean RM richness is shared across quartiles within each cell. Quartiles are not separately recentered. The final cells display the independent four-parameter mixture fits.

Dotted lines mark the RM > 20 support boundary; dashed lines mark zero residual. Error bars are marginal 68% Wilson binomial intervals with the measured center held fixed, not SEM or spatial-covariance errors. Histograms use common bin edges across quartiles within each panel.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'local_overdensity/environment_runner.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'local_overdensity/environment_runner.py').exists():
    raise FileNotFoundError('Set ROOT to DESI_c3clusters')
sys.path.insert(0, str(ROOT))
from local_overdensity.plot_environment_richness import plot_residual_quartiles
INPUT = ROOT / 'local_overdensity/dr9_outputs/section4/dr9/environment.fits'
SAMPLE = ROOT / 'catalogs/conditional_richness/overnight_sample.npz'
HISTOGRAM_BINS = 8


In [ ]:
OUTPUT = ROOT / 'plots' / ('environment_residual_quartiles_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
fig, cdf_fig, summary = plot_residual_quartiles(INPUT, SAMPLE, OUTPUT, HISTOGRAM_BINS)
display(fig)
plt.close(fig)
print('Saved:', OUTPUT)


In [ ]:
display(cdf_fig)
plt.close(cdf_fig)
display(summary)


## Four-parameter mixture fits
Fit parameters are the Gaussian-core location, Gaussian width, intrinsic EMG fraction, and exponential rate. The likelihood is normalized above RM richness 20. Parent-cell mean subtraction is for display only. Inspect convergence before interpreting parameter trends; faded trend markers indicate failed checks. The component curves use the full mixture normalization, so they sum to the total model.


In [ ]:
import pandas as pd
from IPython.display import Image
FITS = ROOT / 'catalogs/environment_mixture_4param_20261027'
FIT_PLOTS = ROOT / 'plots/environment_mixture_4param_20261027'
display(pd.read_csv(FIT_PLOTS / 'fit_statistics.csv'))
display(pd.read_csv(FIT_PLOTS / 'posterior_parameters.csv'))
for q in range(1, 5):
    display(Image(filename=str(FIT_PLOTS / f'Q{q}_mixture_residuals.png')))
display(Image(filename=str(FIT_PLOTS / 'parameter_trends.png')))


In [ ]:
CELL = 'z1_s1_Q1'  # z1-z3, s1-s3, Q1-Q4
display(Image(filename=str(FIT_PLOTS / f'{CELL}_contours.png')))


## Adaptive highest-richness test: pooled redshift and environment
One mixture fit to all clusters in the highest richness bin, over 0.10 <= z < 0.35 and all environments. Choose the four-parameter free-core test or the three-parameter fixed-global-mean test below. In the latter, mu_core_i = global_mean_i - f/tau: global prediction uncertainty is not propagated. These figures use RM richness directly, with no mean subtraction. Stopping requires chain-length and Rhat checks plus stable autocorrelation estimates at two consecutive checkpoints. Convergence does not establish model adequacy.


In [ ]:
import json
RUN = 'environment_mixture_fixed_global_highest_20261027'
# Previous free-core test: RUN = 'environment_mixture_highest_pooled_adaptive_20261027'
POOLED_FIT = ROOT / 'catalogs' / RUN / 'pooled_s3_Q0'
POOLED_PLOTS = ROOT / 'plots' / RUN
if RUN == 'environment_mixture_fixed_global_highest_20261027':
    print('Original 1000-step burn failed convergence. These diagnostics discard 50000 steps after an observed mode transition; original chains/reports are preserved.')
    POOLED_FIT = POOLED_FIT / 'diagnostics_burn_50000'
    POOLED_PLOTS = ROOT / 'plots' / (RUN + '_burn50000')
display(pd.DataFrame(json.loads((POOLED_FIT / 'convergence_history.json').read_text())))
display(pd.read_csv(POOLED_PLOTS / 'posterior_parameters.csv'))
for name in ['pooled_RM_distribution', 'adaptive_convergence', 'pooled_contours', 'ensemble_traces']:
    display(Image(filename=str(POOLED_PLOTS / f'{name}.png')))


## Differential-evolution fits: richness tertiles and redshift
Nine independent three-parameter mixture fits, with all environments pooled. Richness edges are common global equal-count tertiles, not separately recomputed in each redshift bin. The fixed global arithmetic mean determines each object's core location after subtracting f/tau. Global-posterior uncertainty is not propagated. Chains use 80% differential-evolution and 20% snooker proposals. Faded trend points have not passed all stopping criteria. Posterior predictive adequacy is distinct from convergence.


In [ ]:
GRID_PLOTS = ROOT / 'plots/mixture_spec_tertiles_z_de_20261027'
display(pd.read_csv(GRID_PLOTS / 'convergence_and_ppc.csv'))
display(pd.read_csv(GRID_PLOTS / 'posterior_parameters.csv'))
display(Image(filename=str(GRID_PLOTS / 'mixture_posterior_trends.png')))
COMPONENT_PLOTS = ROOT / 'plots/mixture_spec_tertiles_z_de_20261028_components'
# Posterior-mean weighted components sum to the total; band is 16th-84th percentile.
display(Image(filename=str(COMPONENT_PLOTS / 'conditional_cell_distributions.png')))
# Change z1-z3 and s1-s3 to inspect each posterior contour plot.
CONTOUR_CELL = 'z3_s3_Q0'
display(Image(filename=str(GRID_PLOTS / 'contours' / f'{CONTOUR_CELL}_contours.png')))


## Four richness quartiles: pooled redshift, new priors
All environments and 0.10 <= z < 0.35 pooled. Beta(2,2) on f_proj; uniform sigma_lambda in (0,30), with a numerical floor of 1e-12; log-uniform 1/tau in (0.1,500). The log-sigma sampling Jacobian is included. Fixed global arithmetic means and RM>20 truncation are unchanged. Components are posterior-mean weighted contributions, summing to the total. Model fit and convergence must be assessed separately.


In [ ]:
QUARTILE_PLOTS = ROOT / 'plots/mixture_spec_quartiles_beta22_20261028'
display(pd.read_csv(QUARTILE_PLOTS / 'convergence_and_ppc.csv'))
display(pd.read_csv(QUARTILE_PLOTS / 'posterior_parameters.csv'))
for name in ['conditional_cell_distributions', 'mixture_posterior_trends']:
    display(Image(filename=str(QUARTILE_PLOTS / f'{name}.png')))
SPEC_BIN = 4  # 1, 2, 3, or 4; redshift is pooled
display(Image(filename=str(QUARTILE_PLOTS / 'contours' / f'z1_s{SPEC_BIN}_Q0_contours.png')))


## Measurements only: four richness bins, four environment quartiles
Redshift pooled over 0.10 <= z < 0.35. Gaussian-environment quartiles are recalculated within each richness bin. Each distribution is normalized separately; errors are 68% Wilson count intervals. Small horizontal offsets distinguish overlapping markers but do not change bin edges. Redshift and within-bin richness distributions have not been matched. No new models fitted.


In [ ]:
ENV_DATA = ROOT / 'plots/environment_spec_quartiles_20261028_data'
display(Image(filename=str(ENV_DATA / 'RM_environment_quartiles_data.png')))
display(pd.read_csv(ENV_DATA / 'cell_summary.csv'))
